# Prepare OpenBookQA for prompt optimization

This notebook loads the official OpenBookQA files, shows five examples, removes exact training duplicates, and creates a fixed 4,552/900/500 train/validation/test split. The expanded validation set contains the official 500 validation questions plus 400 answer-label-stratified training questions. It also creates three fixed, stratified validation folds of 300 questions for stable prompt selection.

In [ ]:
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

In [ ]:
def find_repo_root():
    """Find the repository root from the current notebook directory."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


def read_jsonl(path):
    """Read non-empty JSON objects from a JSONL file."""
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def show_examples(records, count=5):
    """Display a small number of records in readable JSON."""
    print(json.dumps(records[:count], ensure_ascii=False, indent=2))

In [ ]:
REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data/openbookqa/original/OpenBookQA-V1-Sep2018/Data/Additional"
OUTPUT_DIR = REPO_ROOT / "data/processed/openbookqa"
SUPPLEMENTAL_VALIDATION_SIZE = 400
VALIDATION_FOLD_COUNT = 3
SPLIT_SEED = 42

RAW_SPLITS = {
    "train": RAW_DIR / "train_complete.jsonl",
    "validation": RAW_DIR / "dev_complete.jsonl",
    "test": RAW_DIR / "test_complete.jsonl",
}

raw_records = {split: read_jsonl(path) for split, path in RAW_SPLITS.items()}
for split, records in raw_records.items():
    print(f"{split}: {len(records):,} records")

## Five original examples

In [ ]:
show_examples(raw_records["train"], count=5)

In [ ]:
def normalize_openbookqa(row, split):
    """Convert one OpenBookQA record to the prompt-optimization schema."""
    question = row["question"]
    choices = [
        {"label": choice["label"], "text": choice["text"]}
        for choice in question["choices"]
    ]
    choice_text = {choice["label"]: choice["text"] for choice in choices}
    answer = row["answerKey"]
    return {
        "id": row["id"],
        "dataset": "openbookqa",
        "task_type": "multiple_choice_qa",
        "split": split,
        "question": question["stem"],
        "choices": choices,
        "answer": answer,
        "answer_text": choice_text[answer],
        "fact": row.get("fact1", ""),
    }


def normalize_text(text):
    """Normalize text for reliable duplicate comparison."""
    return " ".join(text.split()).casefold()


def openbookqa_example_key(record):
    """Create a duplicate key from the question and ordered choices."""
    choices = tuple(
        (normalize_text(choice["label"]), normalize_text(choice["text"]))
        for choice in record["choices"]
    )
    return normalize_text(record["question"]), choices


def deduplicate_openbookqa(records):
    """Keep the first exact question-and-choices record."""
    unique_records = []
    first_record_by_key = {}
    for record in records:
        key = openbookqa_example_key(record)
        first_record = first_record_by_key.get(key)
        if first_record is not None:
            if first_record["answer"] != record["answer"]:
                raise ValueError(f"Conflicting answers for duplicate: {record['question']}")
            continue
        first_record_by_key[key] = record
        unique_records.append(record)
    return unique_records, len(records) - len(unique_records)


def select_supplemental_validation(records, selection_size, seed):
    """Move an equal number of A, B, C, and D records into validation."""
    groups = defaultdict(list)
    for record in records:
        groups[record["answer"]].append(record)
    if selection_size % len(groups) != 0:
        raise ValueError("Supplemental validation size must divide across labels.")

    random_generator = random.Random(seed)
    per_label = selection_size // len(groups)
    selected = []
    for label in sorted(groups):
        group_records = list(groups[label])
        random_generator.shuffle(group_records)
        if len(group_records) < per_label:
            raise ValueError(f"Not enough training records for answer label {label}.")
        selected.extend(group_records[:per_label])

    selected_ids = {record["id"] for record in selected}
    remaining = [record for record in records if record["id"] not in selected_ids]
    for record in selected:
        record["split"] = "validation"
        record["validation_source"] = "training_supplement"
    return remaining, selected


def assign_validation_folds(records, fold_count, seed):
    """Create equal folds balanced by validation source and answer label."""
    if len(records) % fold_count != 0:
        raise ValueError("Validation records must divide evenly across folds.")
    groups = defaultdict(list)
    for record in records:
        groups[(record["validation_source"], record["answer"])].append(record)

    random_generator = random.Random(seed)
    folds = [[] for _ in range(fold_count)]
    next_tie_fold = 0
    for key in sorted(groups):
        group_records = list(groups[key])
        random_generator.shuffle(group_records)
        base_size, remainder = divmod(len(group_records), fold_count)
        group_fold_sizes = [base_size] * fold_count
        ranked_folds = sorted(
            range(fold_count),
            key=lambda index: (len(folds[index]), (index - next_tie_fold) % fold_count),
        )
        for fold_index in ranked_folds[:remainder]:
            group_fold_sizes[fold_index] += 1
        next_tie_fold = (next_tie_fold + remainder) % fold_count

        offset = 0
        for fold_index, fold_size in enumerate(group_fold_sizes):
            selected = group_records[offset : offset + fold_size]
            for record in selected:
                record["validation_fold"] = fold_index + 1
            folds[fold_index].extend(selected)
            offset += fold_size

    expected_fold_size = len(records) // fold_count
    if any(len(fold) != expected_fold_size for fold in folds):
        raise ValueError("Could not create equal stratified validation folds.")
    for fold in folds:
        fold.sort(key=lambda record: record["id"])
    return folds

In [ ]:
prepared = {
    split: [normalize_openbookqa(row, split) for row in rows]
    for split, rows in raw_records.items()
}
prepared["train"], removed_train_duplicates = deduplicate_openbookqa(prepared["train"])
assert removed_train_duplicates == 5
for record in prepared["validation"]:
    record["validation_source"] = "official"
prepared["train"], supplemental_validation = select_supplemental_validation(
    prepared["train"], SUPPLEMENTAL_VALIDATION_SIZE, SPLIT_SEED
)
prepared["validation"].extend(supplemental_validation)
prepared["validation"].sort(key=lambda record: record["id"])
validation_folds = assign_validation_folds(
    prepared["validation"], VALIDATION_FOLD_COUNT, SPLIT_SEED
)
print(f"Removed {removed_train_duplicates} exact training duplicates.")
print(f"Moved {len(supplemental_validation)} training questions into validation.")

In [ ]:
def validate_openbookqa(records_by_split, validation_folds):
    """Check sizes, fields, uniqueness, overlap, and validation folds."""
    expected_counts = {"train": 4552, "validation": 900, "test": 500}
    assert {key: len(value) for key, value in records_by_split.items()} == expected_counts
    all_records = [record for records in records_by_split.values() for record in records]
    assert len({record["id"] for record in all_records}) == len(all_records)
    all_keys = [openbookqa_example_key(record) for record in all_records]
    assert len(set(all_keys)) == len(all_keys)
    assert [len(fold) for fold in validation_folds] == [300, 300, 300]
    assert Counter(record["validation_fold"] for record in records_by_split["validation"]) == Counter({1: 300, 2: 300, 3: 300})
    assert Counter(record["validation_source"] for record in records_by_split["validation"]) == Counter({"official": 500, "training_supplement": 400})
    assert all("validation_fold" not in record for record in records_by_split["train"])
    assert all("validation_fold" not in record for record in records_by_split["test"])
    for split, records in records_by_split.items():
        assert all(record["split"] == split for record in records)
        for record in records:
            assert len(record["choices"]) == 4
            labels = {choice["label"] for choice in record["choices"]}
            assert record["answer"] in labels
            assert record["question"].strip() and record["answer_text"].strip()


validate_openbookqa(prepared, validation_folds)
print("Validation passed.")
print({split: len(records) for split, records in prepared.items()})

## Five prepared examples

In [ ]:
show_examples(prepared["validation"], count=5)

In [ ]:
def write_jsonl(path, records):
    """Write records as one JSON object per line."""
    with path.open("w", encoding="utf-8") as stream:
        for record in records:
            stream.write(json.dumps(record, ensure_ascii=False) + "\n")


def write_json(path, value):
    """Write a JSON value with readable indentation."""
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def answer_counts(records):
    """Count answer labels using JSON-friendly sorted keys."""
    return dict(sorted(Counter(record["answer"] for record in records).items()))


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for split, records in prepared.items():
    write_jsonl(OUTPUT_DIR / f"{split}.jsonl", records)
for fold_index, fold_records in enumerate(validation_folds, start=1):
    write_jsonl(OUTPUT_DIR / f"validation_fold_{fold_index}.jsonl", fold_records)

all_records = [record for split in ("train", "validation", "test") for record in prepared[split]]
files = [
    "train.jsonl",
    "validation.jsonl",
    "validation_fold_1.jsonl",
    "validation_fold_2.jsonl",
    "validation_fold_3.jsonl",
    "test.jsonl",
    "all.json",
]
write_json(OUTPUT_DIR / "all.json", all_records)
write_json(
    OUTPUT_DIR / "dataset_info.json",
    {
        "dataset": "openbookqa",
        "task_type": "multiple_choice_qa",
        "splits": {split: len(records) for split, records in prepared.items()},
        "split_answer_counts": {
            split: answer_counts(records) for split, records in prepared.items()
        },
        "validation_fold_answer_counts": {
            str(index): answer_counts(records)
            for index, records in enumerate(validation_folds, start=1)
        },
        "preparation": {
            "official_validation_size": 500,
            "supplemental_validation_size": SUPPLEMENTAL_VALIDATION_SIZE,
            "validation_fold_count": VALIDATION_FOLD_COUNT,
            "validation_fold_size": len(prepared["validation"]) // VALIDATION_FOLD_COUNT,
            "split_seed": SPLIT_SEED,
            "supplement_stratified_by": ["answer"],
            "folds_stratified_by": ["validation_source", "answer"],
            "duplicate_training_rows_removed": removed_train_duplicates,
            "exact_example_overlap_between_splits": 0,
        },
        "files": files,
    },
)
print(f"Saved {len(all_records):,} records and {VALIDATION_FOLD_COUNT} validation folds to {OUTPUT_DIR}")